In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Check For Model Confounders Among Demographic Variables

This **supplementary analysis** checks for potential confounders among demographic variables captured through in the questionaire `Q1` part of the research. 

The goal is to identify whether any of these demographic variables (gender, first-year, first-time programming) significantly impact the the outcomes predicted by the models, and therefore the findings of the research.


The method employed involves:

- Running the key statistical model from the research findings for RQ1, RQ2, and RQ3. as the BASE model.
- Re-running the model including each demographic variable as an additional predictor.
- Evaluation:
    - Is the overall model statistically significant?
    - check p-value of the added demographic variable for statistical significance (p < 0.05).
    - check confidence intervals for the demographic variable to see if they include zero.
    - if significant, assess the effect size, direction of the demographic variable.
    - if significant, does the inclusion of the demographic variable improve model fit (R²)?

In [60]:

df = pd.read_csv('../datasets/dps-thesis-dataset.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87 entries, 0 to 86
Data columns (total 22 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   participant_number                  87 non-null     int64  
 1   survey_done                         87 non-null     object 
 2   chatbot_done                        87 non-null     object 
 3   chatbot_group                       87 non-null     object 
 4   first_year                          78 non-null     object 
 5   gender                              78 non-null     object 
 6   year_in_school_simplified           78 non-null     object 
 7   first_prog_course                   78 non-null     object 
 8   C1                                  87 non-null     float64
 9   C2                                  87 non-null     float64
 10  E1                                  87 non-null     int64  
 11  Difference_C2_C1                    87 non-null

### Engineer Better Column Names 

Let's make dummy coded variables more intuitive for presentation. 

These are best represented as Booleans, so let's make them less confusing.

In [61]:
# Engineer better column names for dummy coded variable presentation. Best as Booleans

df['is_female'] = df['gender_asnumber'].apply(lambda x: 1 if x == 0 else 0)
df['is_first_year'] = df['year_in_school_simplified_asnumber'].apply(lambda x: 1 if x == 0 else 0)
df['is_1st_prog_course'] = df['first_prog_course_asnumber'].apply(lambda x: 1 if x == 1 else 0)

# Checks
#df[['year_in_school_simplified','year_in_school_simplified_asnumber', 'is_first_year']]
#df[['gender','gender_asnumber','is_female']]
#df[['first_prog_course','first_prog_course_asnumber', 'is_1st_prog_course']]

df_survey = df[ df['survey_done'] =='Y']

print("Demographic Check Covariates Dataset Size:", df_survey.shape)

Demographic Check Covariates Dataset Size: (78, 25)


# Summary of Findings At a Glance

**NOTE** `N = 78` Because some participants did not complete Q1.

| Demographic Variable | Dummy-Coded Boolean Column | RQ1 (E1) | RQ2 (E2) | RQ3 (C2) |
| ----- | ----- |   ----- | ----- | ----- |
| Gender | `is_female` (1=Yes, 0=No) | ✅ | ✅ | ❌ |
| First-Year Student | `is_first_year` (1=Yes, 0=No) | ❌ | ❌ | ❌ |
| First Programming Course | `is_1st_prog_course` (1=Yes, 0=No)  | ❌ | ❌ | ❌ |

### Comments On Findings:

- The only potential confounder identified was `gender`, which showed significant effects only with respect to dependent variable `E1`.
- For RQ1 model, Female participants `beta=2.5` R² improved `4.3%`
- For RQ2 model, Female participants in the treatment group `beta=2.9` R² improved `5.3%`
- No other demographic variables showed significant effects for the models.


### Further Explanation of this Effect

A dummy variable regression model `E1 ~ is_female` was performed. (independent t-test equivalent):

- Statically Significant: `p = 0.038`
- 95% Confidence Intervals: Does Not Include Zero `(0.162, 5.665)`
- Effect Size is Small `R² = 0.055`
- Fails to meet Power assumptions. Cohen's `f² = 0.059` (below target `f² >= 0.103`)


In [62]:
import os
import sys
from contextlib import redirect_stdout
import import_ipynb
from helper_functions import linear_regression_analysis, linear_regression_assumption_checks

## Detailed Analysis For Each Research Question

### RQ1

The key model for RQ1 was:

RQ1: `E1 ~ learning_session_count + task_completion_session_count`


In [63]:
checks = ['is_female', 'is_first_year', 'is_1st_prog_course']
ind_vars = ['task_completion_session_count', 'learning_session_count']
dep_var = "E1"

models = {}
with open(os.devnull, 'w') as fnull:
    with redirect_stdout(fnull):
        base = linear_regression_analysis(df_survey, ind_vars, dep_var, do_print=False, do_plot=False)
        for check in checks:
            ind_vars2 = ind_vars + [check]
            out = linear_regression_analysis(df_survey, ind_vars2, dep_var, do_print=False, do_plot=False)
            models[check] = out

print(f"ANALYSIS RQ1: {dep_var} ~ {' + '.join(ind_vars)}\n")
for k, v in models.items():    
    p = v.pvalues[k]
    significant = p < 0.05

    if significant:
        print(f"✅ {k:20s} ==> p-value: {p:0.4f} | Coefficient: {v.params[k]:0.4f}, R² Change: {v.rsquared - base.rsquared:0.4f}")
    else:
        print(f"❌ {k:20s} ==> p-value: {p:0.4f} | Confidence Interval: [{v.conf_int().loc[k,0]:0.4f}, {v.conf_int().loc[k,1]:0.4f}]")



ANALYSIS RQ1: E1 ~ task_completion_session_count + learning_session_count

✅ is_female            ==> p-value: 0.0459 | Coefficient: 2.5762, R² Change: 0.0430
❌ is_first_year        ==> p-value: 0.1360 | Confidence Interval: [-0.6889, 4.9681]
❌ is_1st_prog_course   ==> p-value: 0.1268 | Confidence Interval: [-0.5881, 4.6407]


### RQ2

The key model for RQ2 was:

RQ2: `E1 ~ learning_session_count + task_completion_session_count  + control_treatment_asnumber`


In [64]:
checks = ['is_female', 'is_first_year', 'is_1st_prog_course']
ind_vars = ['task_completion_session_count', 'learning_session_count', 'control_treatment_asnumber']
dep_var = "E1"

models = {}
print("BASE MODEL ANALYSIS RQ2:")
base = linear_regression_analysis(df_survey, ind_vars, dep_var, do_print=True, do_plot=False)
with open(os.devnull, 'w') as fnull:
    with redirect_stdout(fnull):
        for check in checks:
            ind_vars2 = ind_vars + [check]
            out = linear_regression_analysis(df_survey, ind_vars2, dep_var, do_print=False, do_plot=False)
            models[check] = out

print(f"\nCONFOUNDER CHECK RQ2: {dep_var} ~ {' + '.join(ind_vars)}\n")
for k, v in models.items():    
    p = v.pvalues[k]
    significant = p < 0.05

    if significant:
        print(f"✅ {k:20s} ==> p-value: {p:0.4f} | Coefficient: {v.params[k]:0.4f}, R² Change: {v.rsquared - base.rsquared:0.4f}")
    else:
        print(f"❌ {k:20s} ==> p-value: {p:0.4f} | Confidence Interval: [{v.conf_int().loc[k,0]:0.4f}, {v.conf_int().loc[k,1]:0.4f}]")



BASE MODEL ANALYSIS RQ2:

Fitting model with standard OLS...
********************************************************************************
INDEPENDENT VAR: ['task_completion_session_count', 'learning_session_count', 'control_treatment_asnumber']
DEPENDENT VAR  : E1
REGRESSORS     : 3
                            OLS Regression Results                            
Dep. Variable:                     E1   R-squared:                       0.238
Model:                            OLS   Adj. R-squared:                  0.207
Method:                 Least Squares   F-statistic:                     7.712
Date:                Tue, 06 Jan 2026   Prob (F-statistic):           0.000150
Time:                        00:50:52   Log-Likelihood:                -240.65
No. Observations:                  78   AIC:                             489.3
Df Residuals:                      74   BIC:                             498.7
Df Model:                           3                                         
C

### RQ3

The key model for RQ3 was:

RQ2: `E2 ~ learning_session_count + task_completion_session_count  + C1`

In [65]:
checks = ['is_female', 'is_first_year', 'is_1st_prog_course']
ind_vars = ['task_completion_session_count', 'learning_session_count', 'C1']
dep_var = "C2"

models = {}
print("BASE MODEL ANALYSIS RQ3:")
base = linear_regression_analysis(df_survey, ind_vars, dep_var, do_print=True, do_plot=False, robust_std_errors=True)
with open(os.devnull, 'w') as fnull:
    with redirect_stdout(fnull):
        for check in checks:
            ind_vars2 = ind_vars + [check]
            out = linear_regression_analysis(df_survey, ind_vars2, dep_var, do_print=False, do_plot=False, robust_std_errors=True)
            models[check] = out

print(f"\nCONFOUNDER CHECK RQ3: {dep_var} ~ {' + '.join(ind_vars)}\n")
for k, v in models.items():    
    p = v.pvalues[k]
    significant = p < 0.05

    if significant:
        print(f"✅ {k:20s} ==> p-value: {p:0.4f} | Coefficient: {v.params[k]:0.4f}, R² Change: {v.rsquared - base.rsquared:0.4f}")
    else:
        print(f"❌ {k:20s} ==> p-value: {p:0.4f} | Confidence Interval: [{v.conf_int().loc[k,0]:0.4f}, {v.conf_int().loc[k,1]:0.4f}]")



BASE MODEL ANALYSIS RQ3:

Fitting model with robust standard errors (HC3)...
********************************************************************************
INDEPENDENT VAR: ['task_completion_session_count', 'learning_session_count', 'C1']
DEPENDENT VAR  : C2
REGRESSORS     : 3
                            OLS Regression Results                            
Dep. Variable:                     C2   R-squared:                       0.643
Model:                            OLS   Adj. R-squared:                  0.628
Method:                 Least Squares   F-statistic:                     45.63
Date:                Tue, 06 Jan 2026   Prob (F-statistic):           8.35e-17
Time:                        00:50:52   Log-Likelihood:                -193.71
No. Observations:                  78   AIC:                             395.4
Df Residuals:                      74   BIC:                             404.9
Df Model:                           3                                         
Covarianc

### Making Sense of Gender Effects on E1

Let's perform a t-test to see if there is a significant difference in mean `E1` scores among the genders. We will do this with a dummy variable regression model.

- Statically Significant: Yes (p = 0.038)
- Confidence Intervals: Does Not Include Zero (0.162, 5.665)
- However, Effect Size is Small (R² = 0.055)
- Fails to meet Power assumptions. Cohen's f² = 0.059 (below your target of 0.103)
- `is_female` works as a control but not as a primary predictor.

In [69]:
ind_vars = ['is_female']
dep_var = "E1"

print("E1 ~ Gender:")
base = linear_regression_analysis(df_survey, ind_vars, dep_var, do_print=True, do_plot=False)


E1 ~ Gender:

Fitting model with standard OLS...
********************************************************************************
INDEPENDENT VAR: ['is_female']
DEPENDENT VAR  : E1
REGRESSORS     : 1
                            OLS Regression Results                            
Dep. Variable:                     E1   R-squared:                       0.055
Model:                            OLS   Adj. R-squared:                  0.043
Method:                 Least Squares   F-statistic:                     4.448
Date:                Tue, 06 Jan 2026   Prob (F-statistic):             0.0382
Time:                        01:00:31   Log-Likelihood:                -249.04
No. Observations:                  78   AIC:                             502.1
Df Residuals:                      76   BIC:                             506.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
          